In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '2_train_clean2.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '1_val_fe.parquet')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드


# ============================================================
# 0. optuna study파일 db(cat_optuna) 접근, study파일 불러오기
# ============================================================

# import optuna

# study = optuna.load_study(
#     study_name='catboost_2_clean',
#     storage=f"sqlite:///{PROJECT_ROOT / 'models' / 'cat_optuna.db'}"
# )

# print('현재까지 전체 trial 수: ', len(study.trials))
# print('현재까지 최적 trial: ', study.best_trial.number)
# print('현재까지 최적 RMSE: ', study.best_value)
# print('현재까지 최적 hp조합: ', study.best_params)

In [ ]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np
import optuna

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

c:\Users\soohok402\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:
# =============================================================
# 2. train 데이터를 튜닝 검증용으로 재분할(val은 최종 val로 남겨둠)
# =============================================================
# 2-1. train -> train_optuna, val_optuna

# 기존 train의 event_id 확인
print(f"train의 event_id 개수: {train['event_id'].nunique()}")  # 98개

optuna_split_idx = int((train['event_id'].nunique()) * 0.8)
print(f"재분할 train_optuna event_id 개수: {optuna_split_idx}")
print()

train_optuna = train.loc[
    train['event_id'] <= optuna_split_idx
]
val_optuna = train.loc[
    train['event_id'] > optuna_split_idx
]

print(f"전체 train event_id 개수: {train['event_id'].nunique()}")
print(f"train_optuna event_id 개수: {train_optuna['event_id'].nunique()}")
print(f"val_optuna event_id 개수: {val_optuna['event_id'].nunique()}")

train의 event_id 개수: 98
재분할 train_optuna event_id 개수: 78

전체 train event_id 개수: 98
train_optuna event_id 개수: 78
val_optuna event_id 개수: 20


In [5]:
# =============================================================
# 3. 종속변수 설정
# =============================================================

y_train_optuna = train_optuna[target_col]
y_val_optuna = val_optuna[target_col]

In [6]:
print(len(train_optuna['station_id'].unique()))
print(len(val_optuna['station_id'].unique()))
print(
    set(train_optuna['station_id'].unique()) - 
    set(val_optuna['station_id'].unique())
    ) # set() 비어있으면 둘은 같은 집합, OHE 문제 없음

465
465
set()


In [7]:
# ============================================================
# 4. station_id 원핫인코딩 독립변수 생성
# ============================================================

# OHE 전 독립변수 데이터셋
X_train_optuna = train_optuna[num_cols_fe]
X_val_optuna = val_optuna[num_cols_fe]

# 원핫인코딩
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train_optuna[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train_optuna[['station_id']])
val_st_id_ohe = ohe.transform(val_optuna[['station_id']])
# 여기까지는 일반적인 인코딩 과정

# train, val 셋에 합치기
from scipy.sparse import hstack
# scipy안에 희소행렬을 다루는 서브캐키지 scipy.sparse
# hstack(horizontal stack)은 concat(axis=1)과 같은 역할, df가 아닌 행렬도 붙일 수 있는 함수

X_train_optuna_ohe = hstack([
    X_train_optuna,
    train_st_id_ohe
]).tocsr()  # df + 희소행렬(ohe) -> csr형태로 저장하여 sparse 상태 유지
# 이 변수는 더이상 df가 아님

X_val_optuna_ohe = hstack([
    X_val_optuna,
    val_st_id_ohe
]).tocsr()

print("원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 늘어남")
print(X_train_optuna.shape, X_train_optuna_ohe.shape)
print(X_val_optuna.shape, X_val_optuna_ohe.shape)
# 제공된 데이터셋의 관측소 개수는 465개(md파일에 명시)
# 기존컬럼 31개 + 465 = 496이 정상

원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 늘어남
(403990, 31) (403990, 496)
(212309, 31) (212309, 496)


In [ ]:
# ============================================================
# 5-1. optuna1 - objective함수 정의
# ============================================================

# 라이브러리 호출은 위에 해놨음

def objective(trial):

    # 1. 하이퍼파라미터 탐색
    depth = trial.suggest_int(
        'depth',
        4,
        9
    )

    learning_rate = trial.suggest_float(
        'learning_rate',
        0.01,
        0.1,
        log=True
    )

    l2_leaf_reg = trial.suggest_float(
        'l2_leaf_reg',
        1.0,
        10.0,
        log=True
    )

    random_strength = trial.suggest_float(
        'random_strength',
        0.01,
        10.0,
        log=True
    )

    # 2. 모델 정의
    model = CatBoostRegressor(
        iterations=2000,  # 최대 800회 부스팅까지 허용
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        random_strength=random_strength,
        loss_function='RMSE',
        random_seed=42,  # random_state와 동일한 효과
        thread_count=2,  # RF의 n_jobs와 동일: 병렬작업 사용개수
        verbose=200,  # 100개 트리마다 현황정보 표기
        allow_writing_files=False  # 학습 중 로그 등을 별도로 기록, 분석과 무관하므로 불필요
    )

    # 3. 모델 학습
    model.fit(
        X_train_optuna_ohe,
        y_train_optuna,
        eval_set=(X_val_optuna_ohe, y_val_optuna),  # 내부 검증셋
        early_stopping_rounds=100  # RMSE가 100번 연속 개선되지 않으면 종료
    )

    # 4. 검증 데이터 예측
    pred = model.predict(
        X_val_optuna_ohe
    )

    # 5. RMSE 계산
    rmse = root_mean_squared_error(
        y_val_optuna,
        pred
    )

    # 6. Optuna에 RMSE 반환
    return rmse

In [ ]:
# ============================================================
# 5-2. optuna1 - Study 생성 및 실행
# ============================================================

study = optuna.create_study(
    study_name='catboost_2_clean',
    storage=f"sqlite:///{PROJECT_ROOT / 'models' / 'cat_optuna.db'}",
    #sqlite:/// sql db로 다음의 장소에 저장하겠다.
    load_if_exists=True,  # 동일 stuky 존재 시 create X, 불러오기 O
    direction='minimize',
    sampler=optuna.samplers.TPESampler(
        seed=42  # TPESampler의 hp 제안 시 사용하는 난수 고정,
    )            # model의 random_seed와 별도
)

study.optimize(
    objective,
    n_trials=30
)

# 최적 Trial 번호
print('Best Trial:', study.best_trial.number)

# 최저 RMSE
print('Best RMSE:', study.best_value)

# 최적 하이퍼파라미터
print('Best Parameters:', study.best_params)

[I 2026-10-23 16:39:14,290] A new study created in RDB with name: catboost_2_clean


0:	learn: 0.3599802	test: 0.4330085	best: 0.4330085 (0)	total: 148ms	remaining: 4m 55s
200:	learn: 0.1815194	test: 0.3144118	best: 0.3144118 (200)	total: 9.98s	remaining: 1m 29s
400:	learn: 0.1693580	test: 0.3113139	best: 0.3113139 (400)	total: 20.3s	remaining: 1m 20s
600:	learn: 0.1618782	test: 0.3102119	best: 0.3102119 (600)	total: 29.8s	remaining: 1m 9s
800:	learn: 0.1562869	test: 0.3098891	best: 0.3098705 (783)	total: 39.1s	remaining: 58.5s
1000:	learn: 0.1517937	test: 0.3097108	best: 0.3096264 (908)	total: 48.4s	remaining: 48.3s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.3096263835
bestIteration = 908

Shrink model to first 909 iterations.


[I 2026-10-23 16:40:05,064] Trial 0 finished with value: 0.3096263837484288 and parameters: {'depth': 6, 'learning_rate': 0.08927180304353628, 'l2_leaf_reg': 5.395030966670228, 'random_strength': 0.6251373574521749}. Best is trial 0 with value: 0.3096263837484288.


0:	learn: 0.3753093	test: 0.4451076	best: 0.4451076 (0)	total: 39.1ms	remaining: 1m 18s
200:	learn: 0.2304189	test: 0.3393397	best: 0.3393397 (200)	total: 7.11s	remaining: 1m 3s
400:	learn: 0.2137931	test: 0.3287766	best: 0.3287766 (400)	total: 14.4s	remaining: 57.4s
600:	learn: 0.2060058	test: 0.3233764	best: 0.3233764 (600)	total: 21.7s	remaining: 50.5s
800:	learn: 0.2013651	test: 0.3203207	best: 0.3203207 (800)	total: 29.1s	remaining: 43.5s
1000:	learn: 0.1977110	test: 0.3184126	best: 0.3184126 (1000)	total: 36.2s	remaining: 36.1s
1200:	learn: 0.1947205	test: 0.3168815	best: 0.3168815 (1200)	total: 43.2s	remaining: 28.7s
1400:	learn: 0.1920768	test: 0.3157756	best: 0.3157712 (1397)	total: 50.3s	remaining: 21.5s
1600:	learn: 0.1898029	test: 0.3149328	best: 0.3149328 (1600)	total: 57.8s	remaining: 14.4s
1800:	learn: 0.1877110	test: 0.3141853	best: 0.3141848 (1799)	total: 1m 5s	remaining: 7.2s
1999:	learn: 0.1859051	test: 0.3135737	best: 0.3135737 (1999)	total: 1m 12s	remaining: 0us

b

[I 2026-10-23 16:41:19,277] Trial 1 finished with value: 0.3135737361281902 and parameters: {'depth': 4, 'learning_rate': 0.01432169828911152, 'l2_leaf_reg': 1.1430983876313219, 'random_strength': 3.9676050770529883}. Best is trial 0 with value: 0.3096263837484288.


...

0:	learn: 0.3749590	test: 0.4449427	best: 0.4449427 (0)	total: 85.6ms	remaining: 2m 51s
200:	learn: 0.2124134	test: 0.3288073	best: 0.3288073 (200)	total: 15.8s	remaining: 2m 21s
400:	learn: 0.1968288	test: 0.3197775	best: 0.3197775 (400)	total: 32.1s	remaining: 2m 7s
600:	learn: 0.1898421	test: 0.3166171	best: 0.3166171 (600)	total: 48.4s	remaining: 1m 52s
800:	learn: 0.1850760	test: 0.3148122	best: 0.3148122 (800)	total: 1m 4s	remaining: 1m 36s
1000:	learn: 0.1799863	test: 0.3129955	best: 0.3129857 (999)	total: 1m 20s	remaining: 1m 19s
1200:	learn: 0.1758492	test: 0.3117279	best: 0.3117279 (1200)	total: 1m 35s	remaining: 1m 3s
1400:	learn: 0.1723654	test: 0.3110728	best: 0.3110612 (1397)	total: 1m 50s	remaining: 47.1s
1600:	learn: 0.1694055	test: 0.3103832	best: 0.3103832 (1600)	total: 2m 5s	remaining: 31.2s
1800:	learn: 0.1668107	test: 0.3098542	best: 0.3098542 (1800)	total: 2m 20s	remaining: 15.5s
1999:	learn: 0.1644909	test: 0.3094843	best: 0.3094843 (1999)	total: 2m 35s	remaining

[I 2026-10-23 17:40:48,308] Trial 28 finished with value: 0.3094842631314059 and parameters: {'depth': 8, 'learning_rate': 0.015595868911660312, 'l2_leaf_reg': 6.125474549149134, 'random_strength': 7.905171372195081}. Best is trial 25 with value: 0.30690033597891314.


0:	learn: 0.3707396	test: 0.4413126	best: 0.4413126 (0)	total: 127ms	remaining: 4m 12s
200:	learn: 0.1884745	test: 0.3170956	best: 0.3170956 (200)	total: 22.5s	remaining: 3m 20s
400:	learn: 0.1752140	test: 0.3125677	best: 0.3125677 (400)	total: 45.2s	remaining: 3m
600:	learn: 0.1663246	test: 0.3106077	best: 0.3105905 (596)	total: 1m 6s	remaining: 2m 35s
800:	learn: 0.1600053	test: 0.3095998	best: 0.3095878 (798)	total: 1m 28s	remaining: 2m 12s
1000:	learn: 0.1548039	test: 0.3090590	best: 0.3090590 (1000)	total: 1m 49s	remaining: 1m 49s
1200:	learn: 0.1504673	test: 0.3086148	best: 0.3085974 (1198)	total: 2m 11s	remaining: 1m 27s
1400:	learn: 0.1467333	test: 0.3083802	best: 0.3083762 (1389)	total: 2m 33s	remaining: 1m 5s
1600:	learn: 0.1434653	test: 0.3082230	best: 0.3082230 (1600)	total: 2m 54s	remaining: 43.5s
1800:	learn: 0.1404709	test: 0.3081428	best: 0.3080921 (1763)	total: 3m 16s	remaining: 21.7s
1999:	learn: 0.1378063	test: 0.3080781	best: 0.3080571 (1971)	total: 3m 37s	remaining

[I 2026-10-23 17:44:28,793] Trial 29 finished with value: 0.30805707337828786 and parameters: {'depth': 9, 'learning_rate': 0.035778680589080965, 'l2_leaf_reg': 5.516502488122916, 'random_strength': 4.646811026053584}. Best is trial 25 with value: 0.30690033597891314.


Best Trial: 25
Best RMSE: 0.30690033597891314
Best Parameters: {'depth': 9, 'learning_rate': 0.029074177663135446, 'l2_leaf_reg': 4.928912168955859, 'random_strength': 6.6331205928943495}


In [14]:
# Catboost RMSE가 0.2 -> 0.3 이유를 확인하고자 함
# train_optuna, val_optuna, val(기존)
y_train = train[target_col]
y_val = val[target_col]

r = [0, 0.5, 0.9, 0.95, 0.99]

stats_list = pd.concat(
    [
        y_train_optuna.describe(
            percentiles=r
        ).rename('Train_optuna'),

        y_val_optuna.describe(
            percentiles=r
        ).rename('Val_optuna'),

        y_train.describe(
            percentiles=r
        ).rename('Train'),

        y_val.describe(
            percentiles=r
        ).rename('Val')
    ],
    axis=1
)

print(stats_list.to_csv(sep='\t'))

	Train_optuna	Val_optuna	Train	Val
count	403990.0	212309.0	616299.0	133325.0
mean	0.09959050966608084	0.13476070256089004	0.11170629840385918	0.0964470279392462
std	0.37788368906672337	0.4458550321811159	0.4029433452057842	0.35729989589083766
min	-2.18	-2.07	-2.18	-2.12
0%	-2.18	-2.07	-2.18	-2.12
50%	0.01	0.01	0.01	0.01
90%	0.23	0.38	0.28	0.24
95%	0.49	0.78	0.5809999999997671	0.51
99%	1.76	2.19	1.94	1.56
max	7.98	8.3	8.3	7.71



In [ ]:
print(study.best_params)

{'depth': 9, 'learning_rate': 0.029074177663135446, 'l2_leaf_reg': 4.928912168955859, 'random_strength': 6.6331205928943495}


In [ ]:
# ============================================================
# 6. iteration 3000으로 단일모델 한번 돌려보기(optuna셋으로)
# ============================================================

model_iter_3000 = CatBoostRegressor(
    **study.best_params,
    iterations=3000,  # 최대 3000회 부스팅까지 허용
    loss_function='RMSE',
    random_seed=42,  # random_state와 동일한 효과
    thread_count=2,  # RF의 n_jobs와 동일: 병렬작업 사용개수
    verbose=200,  # 200개 트리마다 현황정보 표기
    allow_writing_files=False  # 학습 중 로그 등을 별도로 기록, 분석과 무관하므로 불필요
)

# 3. 모델 학습
model_iter_3000.fit(
    X_train_optuna_ohe,
    y_train_optuna,
    eval_set=(X_val_optuna_ohe, y_val_optuna),  # 내부 검증셋
    early_stopping_rounds=100  # RMSE가 100번 연속 개선되지 않으면 종료
)

# 4. 검증 데이터 예측
pred_iter_3000 = model_iter_3000.predict(
    X_val_optuna_ohe
)

# 5. RMSE 계산
rmse_iter_3000 = root_mean_squared_error(
    y_val_optuna,
    pred_iter_3000
)

print(f"Validation RMSE: {rmse_iter_3000:.6f}")
print(f"Best iteration: {model_iter_3000.get_best_iteration()}")

0:	learn: 0.3722517	test: 0.4427426	best: 0.4427426 (0)	total: 123ms	remaining: 6m 9s
200:	learn: 0.1939771	test: 0.3185219	best: 0.3185219 (200)	total: 23.8s	remaining: 5m 31s
400:	learn: 0.1813137	test: 0.3137582	best: 0.3137506 (399)	total: 47.8s	remaining: 5m 9s
600:	learn: 0.1718818	test: 0.3108635	best: 0.3108578 (598)	total: 1m 10s	remaining: 4m 42s
800:	learn: 0.1652158	test: 0.3097169	best: 0.3097008 (797)	total: 1m 32s	remaining: 4m 13s
1000:	learn: 0.1599913	test: 0.3087189	best: 0.3087152 (998)	total: 1m 54s	remaining: 3m 49s
1200:	learn: 0.1555753	test: 0.3078578	best: 0.3078578 (1200)	total: 2m 17s	remaining: 3m 25s
1400:	learn: 0.1518334	test: 0.3075244	best: 0.3075229 (1396)	total: 2m 39s	remaining: 3m 1s
1600:	learn: 0.1484647	test: 0.3073166	best: 0.3072940 (1581)	total: 3m 1s	remaining: 2m 38s
1800:	learn: 0.1455547	test: 0.3070715	best: 0.3070589 (1758)	total: 3m 23s	remaining: 2m 15s
2000:	learn: 0.1428051	test: 0.3069124	best: 0.3069003 (1985)	total: 3m 45s	remain